# Unit 2 - Applied PCA
In this practical you will learn:
* Python idioms for 
 * plotting with matplotlib,
 * array addressing with numpy,
* The general scikit-learn workflow
* Performing PCA with scikit-learn
* Essential Data analysis with PCA and scikit-learn

# 1. Python idioms used in this practical
We will start with a refresher of Python idioms that we use in this practical. You can skip this if you feel you are already deeply familiar with the concepts presented. Feel free to come back to this section in case any code is unclear. 

## 1.1 matplotlib idioms

### plotting with ax.plot(...)
Internally, matplotlib uses an `Axes` object to plot the data in. All plotting can be done via methods of this object. to This object has all the methods one needs for plotting. In addition, it allows to adjust the plot appearance, add labels, etc.

Therefore it is often practical to plot via the `Axes` object rather than using e.g. `plt.plot(...)`

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

ax = plt.gca() #obtains the Axes object
ax.plot([2,5,4,6,3,5,2])

#decorate the plot; set axis labels 
t = ax.set_xlabel('the x variable')
t = ax.set_ylabel('the y variable')

#manually set ticks on the y-axis
t = ax.set_yticks([2,4,6])

# manually set labels
t =  ax.set_yticklabels(['two', 'four', 'six'])

Matplotlib can display square matrices as images via the `imshow` function. A `colorbar` alongside the figure acts as a kind of 'legend' that allows to guess numerical values from the colours. 

In [ ]:
import numpy as np
ax = plt.gca()
matrix = np.array([[3,2,3],
                   [4,6,3],
                   [1,0,1]])
img = ax.imshow(matrix)
c = plt.colorbar(img) #note that this needs to be done via `plt`, not `ax`

## 1.2 numpy idioms
This is essential numpy you will need. Let's use a [Pascal matrix](https://en.wikipedia.org/wiki/Pascal_matrix) as an example: 

In [ ]:
from scipy.linalg import pascal

#make pascal matrix, 5 rows, 5 columns
p = pascal(5)
#slice parts of it to make it non symmetric
p=p[1:4,:]
p

### mean, standard deviation, variance in arrays and matrices
You can compute mean, standard deviation and variance with numpy via `mean`, `std` and `var`. Normally these 'flatten' the array and return only one number:

In [ ]:
np.mean(p)

But you can compute them separately for each column...

In [ ]:
np.mean(p, axis=0)

... or for each row:

In [ ]:
np.mean(p, axis=1)

We mostly compute them separately for each column, as we are interested in mean and variance of features.

In [ ]:
np.std(p, axis=0)

In [ ]:
np.var(p, axis=0)

### Slicing via the `:` operator 
Quite often we will only want to use parts of an array. This is accomplished by 'slicing' via the `:` operator. By itself, it selects all values in one dimension:

In [ ]:
p[2,:] # all columns in row two. counting starts at zero!

In [ ]:
p[:,1] # all rows in column 1. 

Notice that a column is selected, but the resulting array is printed as a row!

The `:` operator can be used to select ranges of data:

In [ ]:
p[2:4, 3:5] # data in rows 2 and 3, and columns 3 and 4 

### array indexing with booleans
You can apply a criterion to a range of values and use the resulting boolean array to address the columns and rows. 

For example, lets check which columns have variance larger than 8.0:

In [ ]:
cols = np.var(p, axis=0) > 8.0
cols

We can use this to select those columns from the array:

In [ ]:
p[:, cols]

Of course array contents can be addressed with numerical indices, too:

In [ ]:
p[:, [2,3,4]] #selects columns 2,3,4 

### boolean arithmetics for indexing
Boolean index arrays can be combined using boolean arithmetics. For example, consider the following array:

In [ ]:
a = np.array([3,7,5,8,6,2,1,9,4,5,7,8,2,3,6,4,7,8,3,4,5])

Lets place a criterion that values should be greater than 3:

In [ ]:
gt3 = a > 3
gt3

In addition, values should be smaller than 8:

In [ ]:
lt8 = a < 8
lt8

Finally, let's combine those two criteria using the boolean 'and':

In [ ]:
crit = np.logical_and(lt8,gt3)
crit

... and we can finally select those values from the array which are greater than 3 and less than 8:

In [ ]:
a[crit]

## 1.3 sklearn general workflow
Scikit-learn provides machine learning algorithms, called *estimators*, and transformers and preprocessors (like to normalise data.  All of these support a `fit` and a `transform` method. 
* `fit` 'trains' the machine learning model, or extracts the parameters for the preprocessor.
* `transform` applies the learnt model or parameterised preprocessing to the data.
    
For example, the `StandardScaler` that scales a features matrix to zero mean and unit variance.

Let's look at the pascal matrix as an example again, this time using the "upper triangular' form:

In [ ]:
p = pascal(3, 'upper', exact=False) 
#exact=False creates a floating point instead of an int array
#avoids an error message further down.

print('before scaling:')
print(p)
print('per-column mean:')
print('{}'.format(np.mean(p, axis=0)))
print('per-column variance:')
print('{}'.format(np.var(p, axis=0)))

Now we use `StandardScaler` to scale the matrix:

In [ ]:
from sklearn import preprocessing as pp
# sklearn standard scaler in action
scaler = pp.StandardScaler()
scaler.fit(p) # 'learns' mean and variance of a
p_dash = scaler.transform(p) # 'transforms' a

print()
print('after scaling:')
print(p_dash)
print('per-column mean:')
print('{}'.format(np.mean(p_dash, axis=0)))
print('per-column variance:')
print('{}'.format(np.var(p_dash, axis=0)))

Note how the normalised mean is vanishingly small. 

# 2. PCA essentials 

Let's do a PCA on an anisotropic gaussian blob of data points. 

First we make the data and plot it. We use the `multivariate_normal` function of nupy that generates a gaussian blob with a given covariance matrix - very convenient!

In [ ]:
design_covmat = np.array([[5.,4.],
                          [4.,6.]]) #the covariance matrix
center = np.array([1,1.5]) # the center of the blob
blob = np.random.multivariate_normal(mean=center, 
                                     cov=design_covmat, 
                                     size=100)

#plotting
ax = plt.gca()
ax.scatter(blob[:,0], blob[:,1])
ax.axis('scaled')
t = ax.set_xlabel('x')
t = ax.set_ylabel('y')

Next step: do a PCA with scikit-learn.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(blob)
blob_pca = pca.transform(blob)

#plotting
ax = plt.gca()
ax.scatter(blob_pca[:,0], blob_pca[:,1])
ax.axis('scaled')
t = ax.set_xlabel('PC 1')
t = ax.set_ylabel('PC 2')

Notice two things:
1. the center of the blob is now at the origin ([0.,0.])
2. the first component (i.e. x-axis) is aligned in the direction of highest variance. 

### Covariance matrix from PCA
The PCA implementation in scikit-learn gives us access to the covariance matrix. Let's compare it to the covariance matrix that we used to make the blobs:

In [ ]:
computed_covmat = pca.get_covariance()
print("covmat from PCA:")
print("{}".format(computed_covmat))
print()
print("design covmat:")
print("{}".format(design_covmat))


**Question 1 and 2:**
* Are the design and the computed covariance matrices identical? 
* If not, why? What does it mean for using PCA in applied data analysis?

**Task 1**
* Increase the number of points in the blob from 100 to 10000. Repeat the analysis. How large is the difference between the observed and design covariance matrix now? Can you explain the difference to the previous example with the 100 points in the blob?

## 2.1 PCA on iris

We're just warming up so let's reproduce the Iris analysis from the lecture. 

We start by loading the iris data set, which is conveniently contained in scikit-learn:

In [ ]:
import sklearn.datasets as data
iris = data.load_iris()

The result is a data structure, in the form of a dictionary. let's explore it:

In [ ]:
iris.keys()

The names of the keys are pretty much self-explanatory. Have a look at the `DESCR` key for more information about the data set:

In [ ]:
print(iris['DESCR'])

We can construct a pandas data frame from that information:

In [ ]:
import pandas as pd
iris_df = pd.DataFrame(iris['data'], columns=iris['feature_names'])
iris_df['species'] = iris['target']
iris_df

### **Task 2:**
Construct the data frame such that the "species" column has the actual species names (setosa etc).

### PCA on iris

We can essentially use the same approach when out data is in a data frame, we just have to get it out:

In [ ]:
iris_values = iris_df[iris['feature_names']].values
iris_values

In [ ]:
pca = PCA()
pca.fit(iris_values)
iris_pca = pca.transform(iris_values)

#plotting
ax = plt.gca()
ax.scatter(iris_pca[:,0], iris_pca[:,1])
t = ax.set_xlabel('PC1')
t = ax.set_ylabel('PC2')

**Task 3:** color the dots according to species.

### covariance matrix
Let's plot the covariance matrix as numbers:

In [ ]:
pca.get_covariance()

Now let's plot the covariance matrix as an image:

In [ ]:
ax = plt.gca()
covmat = pca.get_covariance()
img = ax.imshow(covmat, cmap='seismic', vmin=-3.2, vmax=3.2)
plt.colorbar(img, label='covariance')
ax.set_yticks([0,1,2,3])
t = ax.set_yticklabels(iris['feature_names'])
ax.set_xticks([0,1,2,3])
t = ax.set_xticklabels(iris['feature_names'], rotation=90)
for x in range(4):
    for y in range(4):
        ax.text(x,y,"{:.2f}".format(covmat[y,x]), {'ha':'center', 'color':"blue"})

In the practicals to follow, we will often visualise matrices as "images", with a colorbar that reveals the numerical value. Usually we will not plot the actual values, resulting in such a plot:

In [ ]:
ax = plt.gca()
img = ax.imshow(covmat, cmap='seismic', vmin=-3.2, vmax=3.2)
plt.colorbar(img, label='covariance')
ax.set_yticks([0,1,2,3])
ax.set_yticks([0,1,2,3])
t = ax.set_yticklabels(iris['feature_names'])
ax.set_xticks([0,1,2,3])
t = ax.set_xticklabels(iris['feature_names'], rotation=90)


### Scree plot
The scree plot displays the amount of explained variance.

In [ ]:
ax = plt.gca()
expl_var = pca.explained_variance_ratio_
ax.plot(expl_var, marker='x')
ax.set_ylim(0,1.)
ax.set_xticks([0,1,2,3])
ax.set_xticklabels(["PC{}".format(i) for i in range(4)])

### Components
The principal components are a weighted sum of the data vectors. Let's retrieve them from the PCA:

In [ ]:
comps = pca.components_
ax = plt.gca()
img = ax.imshow(comps, cmap='seismic', vmin=-1, vmax=1)
plt.colorbar(img, label="weight")
ax.set_yticks([0,1,2,3])
t = ax.set_yticklabels(["PC{}".format(i+1) for i in range(4)])
ax.set_xticks([0,1,2,3])
t = ax.set_xticklabels(iris['feature_names'], rotation=90)

**Task 4**: Which features have a positive weight on PC1?

### Normalisation
Finally, let's normalise the data and do the PCA again:

In [ ]:
ss = pp.StandardScaler()
ss.fit(iris_values)
pca_norm = PCA()
pca_norm.fit(ss.transform(iris_values)) # note the composition of the commands
iris_pca_norm = pca_norm.transform(ss.transform(iris_values))

#plotting
ax = plt.gca()
ax.scatter(iris_pca_norm[:,0], iris_pca_norm[:,1])
t = ax.set_xlabel('PC1')
t = ax.set_ylabel('PC2')

The difference in result is subtle in the plot of the first two PCs, but it is more drastic in the covariance matrix.

### Task 5, putting it all together:
1. plot the covariance matrix for the normalised data and compare it to that of the raw data.
2. plot the scree-plot for the normalised data and compare it to that of the raw data. 
3. Observe the difference! Compare to the lecture, where we discussed the "variance bug" of PCA.



# Conclusion
* PCA is a powerful tool to get a first glance at data. 
* Normalisation can make a huge difference.
* Dimensionality reduction helps with plotting.
* Scree plot allows to identify how many components are required.
* Component loadings allow interpretation of feature relevance.
* Identifying clusters can help uncover patterns in instances and features. 